In [9]:
import multilayerGM as gm

n_nodes = 20
n_layers = 5
p = 0.8
theta = 0.2 
n_sets = 2
dt = gm.dependency_tensors.UniformMultiplex(n_nodes, n_layers, p)
null = gm.DirichletNull(layers=dt.shape[1:], theta=theta, n_sets=n_sets)

partition = gm.sample_partition(dependency_tensor=dt, null_distribution=null)
multinet = gm.multilayer_DCSBM_network(partition, mu=0.1, k_min=2, k_max=10, t_k=-2)

In [10]:
print("Generated multilayer network with", multinet.number_of_edges(), "edges", multinet.number_of_nodes(), "nodes")

Generated multilayer network with 182 edges 100 nodes


In [ ]:
import networkx as nx
import numpy as np
import pandas as pd

def save_graph_csv_poisson_int_ts(
    G: nx.Graph,
    out_csv: str,
    lam: float = 10.0,
    comm_attr: str = "mesoset",
    seed: int | None = 42,
    base_time: int = 0,
    sort_by_time: bool = True,
    make_strictly_increasing: bool = False,
):
    rng = np.random.default_rng(seed)

    edges = []
    if G.is_multigraph():
        for u, v, k in G.edges(keys=True):
            edges.append((u, v))
    else:
        for u, v in G.edges():
            edges.append((u, v))

    m = len(edges)
    df_cols = ["source", "destination", "timestamp", "source_commu", "destination_comm"]
    if m == 0:
        pd.DataFrame(columns=df_cols).to_csv(out_csv, index=False)
        return

    ts = rng.poisson(lam=lam, size=m).astype(np.int64)

    if make_strictly_increasing:
        ts = np.cumsum(ts + 1).astype(np.int64)

    ts = (ts + np.int64(base_time)).astype(np.int64)

    rows = []
    for i, (u, v) in enumerate(edges):
        rows.append({
            "source": u[0],
            "destination": v[0],
            "timestamp": int(ts[i]),
            "source_commu": G.nodes[u].get(comm_attr, None),
            "destination_commu": G.nodes[v].get(comm_attr, None),
        })

    df = pd.DataFrame(rows)

    df["timestamp"] = df["timestamp"].astype("int64")

    if sort_by_time:
        df = df.sort_values(["timestamp"], kind="stable").reset_index(drop=True)

    df.to_csv(out_csv, index=False)

save_graph_csv_poisson_int_ts(
    multinet,
    out_csv="syn_net.csv",
    lam=5.0,
    comm_attr="mesoset",
    seed=42,
    base_time=0,
    sort_by_time=True,
    make_strictly_increasing=True,
)